In [2]:
%pip install -e "/content/drive/MyDrive/BUSI-XAI"

Obtaining file:///content/drive/MyDrive/BUSI-XAI
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for busi-xai (pyproject.toml) ... done
  Created wheel for busi-xai: filename=busi_xai-0.1.0-0.editable-py3-none-any.whl size=1167 sha256=9428e73c3995a6089c95581e6b24788bb41dd618246848fc79ed6feb74390e4e
  Stored in directory: /tmp/pip-ephem-wheel-cache-btmu6cam/wheels/bc/96/6a/14ad5b6a03036f6f181aa62fbcb8d63ac3b6cb1da04fcdcb1e
Successfully built busi-xai


# Load Data

In [1]:
from pathlib import Path
import json
from busi_xai.config import XAI_DIR, XAI_STABILITY_OUTPUT_DIR, XAI_METHODS
import pandas as pd

ARTIFACTS = {
    "original": XAI_DIR / "xai_records.json",
    "faithfulness": XAI_DIR / "xai_faithfulness_records.json",
    "stability": XAI_STABILITY_OUTPUT_DIR / "xai_stability_records.json",
}

for name, path in ARTIFACTS.items():
    print(f"\n=== {name.upper()} ===")
    print(f"Path: {path}")
    print(f"Exists: {path.exists()}")

    with open(path, "r") as f:
        data = json.load(f)

    print(f"Records: {len(data)}")
    print(f"First record keys: {sorted(data[0].keys())}")

    # Load saved artifacts
with open(ARTIFACTS["original"], "r") as f:
    original_records = json.load(f)

with open(ARTIFACTS["faithfulness"], "r") as f:
    faithfulness_records = json.load(f)

with open(ARTIFACTS["stability"], "r") as f:
    stability_records = json.load(f)

# Convert to DataFrames
original_df = pd.DataFrame(original_records)
faithfulness_df = pd.DataFrame(faithfulness_records)
stability_df = pd.DataFrame(stability_records)

# Select only the fields needed for statistical comparison
original_stats = original_df[
    ["dataset_index", "method", "iou", "dice", "runtime_seconds"]
].copy()

faithfulness_stats = faithfulness_df[
    ["dataset_index", "method", "probability_drop"]
].copy()

stability_stats = stability_df[
    ["dataset_index", "method", "spearman_correlation", "mae"]
].copy()

# Rename stability fields to analysis names
stability_stats = stability_stats.rename(
    columns={
        "spearman_correlation": "stability_spearman",
        "mae": "stability_mae",
    }
)

# Merge using the established experimental alignment
stats_df = (
    original_stats
    .merge(
        faithfulness_stats,
        on=["dataset_index", "method"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        stability_stats,
        on=["dataset_index", "method"],
        how="inner",
        validate="one_to_one",
    )
)

# Expected experimental structure
expected_methods = set(XAI_METHODS)
expected_rows = 97 * len(expected_methods)

print("=== Paired statistical-analysis dataset ===")
print(f"Rows: {len(stats_df)}")
print(f"Unique dataset indices: {stats_df['dataset_index'].nunique()}")
print(f"Methods: {sorted(stats_df['method'].unique())}")
print(f"Expected rows: {expected_rows}")
print(f"Missing values: {int(stats_df.isna().sum().sum())}")

print("\nRows by method:")
print(stats_df["method"].value_counts().sort_index())

print("\nAnalysis columns:")
print(stats_df.columns.tolist())


=== ORIGINAL ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/xai/xai_records.json
Exists: True
Records: 291
First record keys: ['correct', 'dataset_index', 'dice', 'explanation_pixels', 'ground_truth_pixels', 'image_id', 'intersection_pixels', 'iou', 'localisation_percentile', 'method', 'predicted_class', 'predicted_probability', 'runtime_seconds', 'true_label', 'union_pixels']

=== FAITHFULNESS ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/xai/xai_faithfulness_records.json
Exists: True
Records: 291
First record keys: ['baseline_value', 'correct', 'dataset_index', 'deleted_fraction', 'deleted_pixels', 'deletion_percentile', 'image_id', 'method', 'original_probability', 'perturbed_probability', 'predicted_class', 'probability_drop', 'total_pixels', 'true_label']

=== STABILITY ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/xai/stability/xai_stability_records.json
Exists: True
Records: 291
First record keys: ['dataset_index', 'image_id', 'mae', 'method', 'noise_std', 'original_

# Friedman test

In [8]:
from scipy.stats import friedmanchisquare

STATISTICAL_METRICS = [
    "iou",
    "dice",
    "probability_drop",
    "stability_spearman",
    "stability_mae",
    "runtime_seconds",
]

methods_order = ["gradcam", "lime", "shap"]

friedman_results = []

for metric in STATISTICAL_METRICS:
    # Pivot so each row is one test image and each column is one method
    paired = (
        stats_df
        .pivot(
            index="dataset_index",
            columns="method",
            values=metric,
        )
        .loc[:, methods_order]
    )

    statistic, p_value = friedmanchisquare(
        paired["gradcam"],
        paired["lime"],
        paired["shap"],
    )

    friedman_results.append({
        "metric": metric,
        "n": len(paired),
        "friedman_statistic": statistic,
        "p_value": p_value,
    })

friedman_df = pd.DataFrame(friedman_results)

print("=== Friedman overall method comparisons ===")
print(friedman_df.to_string(index=False))

=== Friedman overall method comparisons ===
            metric  n  friedman_statistic      p_value
               iou 97            5.119171 7.733679e-02
              dice 97            5.119171 7.733679e-02
  probability_drop 97           38.185567 5.106337e-09
stability_spearman 97          114.391753 1.445928e-25
     stability_mae 97          162.742268 4.581055e-36
   runtime_seconds 97          194.000000 7.471972e-43


# Wilcoxon signed-rank test

In [3]:
from scipy.stats import wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from busi_xai.config import XAI_METHODS

POSTHOC_METRICS = [
    "probability_drop",
    "stability_spearman",
    "stability_mae",
    "runtime_seconds",
]

method_pairs = list(combinations(XAI_METHODS, 2))

posthoc_results = []

for metric in POSTHOC_METRICS:
    paired = (
        stats_df
        .pivot(
            index="dataset_index",
            columns="method",
            values=metric,
        )
        .loc[:, XAI_METHODS]
    )

    raw_p_values = []
    pair_results = []

    for method_a, method_b in method_pairs:
        statistic, p_value = wilcoxon(
            paired[method_a],
            paired[method_b],
            alternative="two-sided",
        )

        pair_results.append({
            "metric": metric,
            "method_a": method_a,
            "method_b": method_b,
            "n": len(paired),
            "wilcoxon_statistic": statistic,
            "raw_p_value": p_value,
        })

        raw_p_values.append(p_value)

    # Holm correction across the three pairwise comparisons
    _, adjusted_p_values, _, _ = multipletests(
        raw_p_values,
        method="holm",
    )

    for result, adjusted_p in zip(pair_results, adjusted_p_values):
        result["holm_adjusted_p_value"] = adjusted_p
        result["significant_after_holm"] = adjusted_p < 0.05
        posthoc_results.append(result)

posthoc_df = pd.DataFrame(posthoc_results)

print("=== Wilcoxon post-hoc comparisons with Holm correction ===")
print(
    posthoc_df[
        [
            "metric",
            "method_a",
            "method_b",
            "n",
            "wilcoxon_statistic",
            "raw_p_value",
            "holm_adjusted_p_value",
            "significant_after_holm",
        ]
    ].to_string(index=False)
)

=== Wilcoxon post-hoc comparisons with Holm correction ===
            metric method_a method_b  n  wilcoxon_statistic  raw_p_value  holm_adjusted_p_value  significant_after_holm
  probability_drop  gradcam     lime 97              1375.0 3.138020e-04           6.276040e-04                    True
  probability_drop  gradcam     shap 97              1534.0 2.433321e-03           2.433321e-03                    True
  probability_drop     lime     shap 97               681.0 1.055350e-09           3.166051e-09                    True
stability_spearman  gradcam     lime 97               216.0 7.605494e-15           1.521099e-14                    True
stability_spearman  gradcam     shap 97              1456.0 9.257380e-04           9.257380e-04                    True
stability_spearman     lime     shap 97                15.0 1.941689e-17           5.825067e-17                    True
     stability_mae  gradcam     lime 97                75.0 1.218013e-16           2.436026e-16      

# Paired effect sizes and direction

In [4]:
effect_size_results = []

for metric in POSTHOC_METRICS:
    paired = (
        stats_df
        .pivot(
            index="dataset_index",
            columns="method",
            values=metric,
        )
        .loc[:, XAI_METHODS]
    )

    for method_a, method_b in method_pairs:
        a = paired[method_a]
        b = paired[method_b]

        difference = a - b

        # Wilcoxon signed-rank test
        statistic, p_value = wilcoxon(
            a,
            b,
            alternative="two-sided",
        )

        # Rank-biserial correlation for paired samples.
        # SciPy's Wilcoxon statistic is the smaller of the
        # positive- and negative-rank sums.
        nonzero = difference[difference != 0]

        if len(nonzero) == 0:
            rank_biserial = 0.0
        else:
            ranks = nonzero.abs().rank(method="average")
            positive_rank_sum = ranks[nonzero > 0].sum()
            negative_rank_sum = ranks[nonzero < 0].sum()

            rank_biserial = (
                positive_rank_sum - negative_rank_sum
            ) / (positive_rank_sum + negative_rank_sum)

        effect_size_results.append({
            "metric": metric,
            "method_a": method_a,
            "method_b": method_b,
            "median_a": a.median(),
            "median_b": b.median(),
            "median_difference_a_minus_b": difference.median(),
            "mean_difference_a_minus_b": difference.mean(),
            "rank_biserial": rank_biserial,
            "holm_adjusted_p_value": posthoc_df.loc[
                (posthoc_df["metric"] == metric)
                & (posthoc_df["method_a"] == method_a)
                & (posthoc_df["method_b"] == method_b),
                "holm_adjusted_p_value",
            ].iloc[0],
        })

effect_size_df = pd.DataFrame(effect_size_results)

print("=== Paired effect sizes and direction ===")
print(
    effect_size_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

=== Paired effect sizes and direction ===
            metric method_a method_b  median_a  median_b  median_difference_a_minus_b  mean_difference_a_minus_b  rank_biserial  holm_adjusted_p_value
  probability_drop  gradcam     lime  0.109066  0.051827                     0.031057                   0.099360       0.421418               0.000628
  probability_drop  gradcam     shap  0.109066  0.349317                    -0.196877                  -0.096747      -0.354513               0.002433
  probability_drop     lime     shap  0.051827  0.349317                    -0.245131                  -0.196108      -0.713444               0.000000
stability_spearman  gradcam     lime  0.762838  0.342420                     0.371079                   0.346782       0.909110               0.000000
stability_spearman  gradcam     shap  0.762838  0.647583                     0.113098                   0.047810       0.387334               0.000926
stability_spearman     lime     shap  0.342420  0.64

# Save results

In [9]:
from busi_xai.config import METRICS_DIR

STATISTICAL_OUTPUT = (
    METRICS_DIR / "xai_statistical_comparison.json"
)

statistical_artifact = {
    "design": {
        "test_type": "paired repeated-measures",
        "n_test_images": 97,
        "methods": XAI_METHODS,
        "alpha": 0.05,
        "overall_test": "Friedman",
        "posthoc_test": "Wilcoxon signed-rank",
        "multiple_comparison_correction": "Holm",
    },
    "friedman_results": friedman_df.to_dict(orient="records"),
    "posthoc_results": posthoc_df.to_dict(orient="records"),
    "effect_size_results": effect_size_df.to_dict(orient="records"),
}

with open(STATISTICAL_OUTPUT, "w") as f:
    json.dump(
        statistical_artifact,
        f,
        indent=2,
    )

print("=== Statistical comparison artifact saved ===")
print(f"Path: {STATISTICAL_OUTPUT}")
print(f"Friedman tests: {len(friedman_df)}")
print(f"Post-hoc comparisons: {len(posthoc_df)}")
print(f"Effect-size comparisons: {len(effect_size_df)}")

=== Statistical comparison artifact saved ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_statistical_comparison.json
Friedman tests: 6
Post-hoc comparisons: 12
Effect-size comparisons: 12


# Examine localization consistency

In [10]:
localisation_consistency = []

for method in methods_order:
    method_df = stats_df[stats_df["method"] == method]

    iou = method_df["iou"]
    dice = method_df["dice"]

    localisation_consistency.append({
        "method": method,
        "n": len(method_df),

        "iou_mean": iou.mean(),
        "iou_median": iou.median(),
        "iou_std": iou.std(ddof=1),
        "iou_q1": iou.quantile(0.25),
        "iou_q3": iou.quantile(0.75),
        "iou_nonzero_proportion": (iou > 0).mean(),

        "dice_mean": dice.mean(),
        "dice_median": dice.median(),
        "dice_std": dice.std(ddof=1),
        "dice_q1": dice.quantile(0.25),
        "dice_q3": dice.quantile(0.75),
        "dice_nonzero_proportion": (dice > 0).mean(),

        "iou_ge_0_10_proportion": (iou >= 0.10).mean(),
        "iou_ge_0_25_proportion": (iou >= 0.25).mean(),
        "dice_ge_0_20_proportion": (dice >= 0.20).mean(),
        "dice_ge_0_50_proportion": (dice >= 0.50).mean(),
    })

localisation_consistency_df = pd.DataFrame(localisation_consistency)

print("=== RQ4 localisation consistency ===")
print(
    localisation_consistency_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

=== RQ4 localisation consistency ===
 method  n  iou_mean  iou_median  iou_std   iou_q1   iou_q3  iou_nonzero_proportion  dice_mean  dice_median  dice_std  dice_q1  dice_q3  dice_nonzero_proportion  iou_ge_0_10_proportion  iou_ge_0_25_proportion  dice_ge_0_20_proportion  dice_ge_0_50_proportion
gradcam 97  0.190214    0.164653 0.158112 0.046711 0.301591                0.845361   0.291171     0.282750  0.217985 0.089253 0.463419                 0.845361                0.649485                0.329897                 0.628866                 0.206186
   lime 97  0.138400    0.076100 0.161191 0.009794 0.230579                0.907216   0.213258     0.141436  0.216106 0.019398 0.374749                 0.907216                0.463918                0.216495                 0.443299                 0.113402
   shap 97  0.176196    0.157238 0.123693 0.073734 0.266962                1.000000   0.281676     0.271747  0.173133 0.137341 0.421421                 1.000000                0.649485  

# Paired localization success

In [11]:
from statsmodels.stats.contingency_tables import mcnemar

localisation_binary_results = []

for method_a, method_b in method_pairs:
    paired = (
        stats_df
        .pivot(
            index="dataset_index",
            columns="method",
            values="iou",
        )
        .loc[:, [method_a, method_b]]
    )

    success_a = paired[method_a] > 0
    success_b = paired[method_b] > 0

    # 2x2 paired contingency table:
    #
    #                 method_b
    #                 fail   success
    # method_a fail
    # method_a success
    #
    table = [
        [
            int((~success_a & ~success_b).sum()),
            int((~success_a & success_b).sum()),
        ],
        [
            int((success_a & ~success_b).sum()),
            int((success_a & success_b).sum()),
        ],
    ]

    result = mcnemar(
        table,
        exact=True,
    )

    localisation_binary_results.append({
        "method_a": method_a,
        "method_b": method_b,
        "n": len(paired),
        "a_success": int(success_a.sum()),
        "b_success": int(success_b.sum()),
        "both_success": int((success_a & success_b).sum()),
        "both_failure": int((~success_a & ~success_b).sum()),
        "a_only_success": int((success_a & ~success_b).sum()),
        "b_only_success": int((~success_a & success_b).sum()),
        "mcnemar_statistic": result.statistic,
        "p_value": result.pvalue,
    })

localisation_binary_df = pd.DataFrame(localisation_binary_results)

print("=== McNemar tests: non-zero localization overlap ===")
print(
    localisation_binary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

=== McNemar tests: non-zero localization overlap ===
method_a method_b  n  a_success  b_success  both_success  both_failure  a_only_success  b_only_success  mcnemar_statistic  p_value
 gradcam     lime 97         82         88            75             2               7              13           7.000000 0.263176
 gradcam     shap 97         82         97            82             0               0              15           0.000000 0.000061
    lime     shap 97         88         97            88             0               0               9           0.000000 0.003906


# Save results

In [12]:
RQ4_LOCALISATION_OUTPUT = (
    METRICS_DIR / "xai_rq4_localisation_consistency.json"
)

rq4_localisation_artifact = {
    "design": {
        "n_test_images": 97,
        "methods": methods_order,
        "primary_localisation_metric": "iou",
        "secondary_localisation_metric": "dice",
        "binary_localisation_definition": "iou > 0",
        "paired_binary_test": "McNemar exact test",
        "alpha": 0.05,
    },
    "descriptive_consistency": (
        localisation_consistency_df.to_dict(orient="records")
    ),
    "mcnemar_results": (
        localisation_binary_df.to_dict(orient="records")
    ),
}

with open(RQ4_LOCALISATION_OUTPUT, "w") as f:
    json.dump(
        rq4_localisation_artifact,
        f,
        indent=2,
    )

print("=== RQ4 localisation consistency artifact saved ===")
print(f"Path: {RQ4_LOCALISATION_OUTPUT}")
print(
    f"Descriptive method records: "
    f"{len(localisation_consistency_df)}"
)
print(
    f"McNemar pairwise comparisons: "
    f"{len(localisation_binary_df)}"
)

=== RQ4 localisation consistency artifact saved ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_rq4_localisation_consistency.json
Descriptive method records: 3
McNemar pairwise comparisons: 3


In [ ]:
# Validate saved RQ4 localisation consistency artifact

with open(RQ4_LOCALISATION_OUTPUT, "r") as f:
    saved_rq4 = json.load(f)

validation_errors = []

# Check top-level structure
expected_keys = {
    "design",
    "descriptive_consistency",
    "mcnemar_results",
}

if set(saved_rq4.keys()) != expected_keys:
    validation_errors.append(
        f"Unexpected top-level keys: {set(saved_rq4.keys())}"
    )

# Check design
design = saved_rq4.get("design", {})

if design.get("n_test_images") != 97:
    validation_errors.append(
        f"Expected 97 test images, got {design.get('n_test_images')}"
    )

if design.get("methods") != methods_order:
    validation_errors.append(
        f"Unexpected methods: {design.get('methods')}"
    )

if design.get("primary_localisation_metric") != "iou":
    validation_errors.append("Primary metric is not IoU")

if design.get("secondary_localisation_metric") != "dice":
    validation_errors.append("Secondary metric is not Dice")

if design.get("binary_localisation_definition") != "iou > 0":
    validation_errors.append(
        "Unexpected binary localisation definition"
    )

if design.get("paired_binary_test") != "McNemar exact test":
    validation_errors.append(
        "Unexpected paired binary test"
    )

# Check descriptive records
descriptive = saved_rq4.get("descriptive_consistency", [])

if len(descriptive) != 3:
    validation_errors.append(
        f"Expected 3 descriptive records, got {len(descriptive)}"
    )

saved_methods = sorted(
    record.get("method")
    for record in descriptive
)

if saved_methods != sorted(methods_order):
    validation_errors.append(
        f"Unexpected descriptive methods: {saved_methods}"
    )

# Check McNemar records
mcnemar = saved_rq4.get("mcnemar_results", [])

if len(mcnemar) != 3:
    validation_errors.append(
        f"Expected 3 McNemar comparisons, got {len(mcnemar)}"
    )

expected_pairs = {
    ("gradcam", "lime"),
    ("gradcam", "shap"),
    ("lime", "shap"),
}

saved_pairs = {
    (
        record.get("method_a"),
        record.get("method_b"),
    )
    for record in mcnemar
}

if saved_pairs != expected_pairs:
    validation_errors.append(
        f"Unexpected McNemar pairs: {saved_pairs}"
    )

# Check required fields
required_descriptive_fields = {
    "method",
    "n",
    "iou_mean",
    "iou_median",
    "iou_std",
    "iou_q1",
    "iou_q3",
    "iou_nonzero_proportion",
    "dice_mean",
    "dice_median",
    "dice_std",
    "dice_q1",
    "dice_q3",
    "dice_nonzero_proportion",
    "iou_ge_0_10_proportion",
    "iou_ge_0_25_proportion",
    "dice_ge_0_20_proportion",
    "dice_ge_0_50_proportion",
}

for record in descriptive:
    missing = required_descriptive_fields - set(record.keys())

    if missing:
        validation_errors.append(
            f"Missing descriptive fields for "
            f"{record.get('method')}: {sorted(missing)}"
        )

required_mcnemar_fields = {
    "method_a",
    "method_b",
    "n",
    "a_success",
    "b_success",
    "both_success",
    "both_failure",
    "a_only_success",
    "b_only_success",
    "mcnemar_statistic",
    "p_value",
}

for record in mcnemar:
    missing = required_mcnemar_fields - set(record.keys())

    if missing:
        validation_errors.append(
            f"Missing McNemar fields for "
            f"{record.get('method_a')} vs "
            f"{record.get('method_b')}: {sorted(missing)}"
        )

print("=== RQ4 saved artifact validation ===")
print(f"Descriptive method records: {len(descriptive)}")
print(f"McNemar pairwise comparisons: {len(mcnemar)}")
print(f"Validation errors: {len(validation_errors)}")

if validation_errors:
    for error in validation_errors:
        print("ERROR:", error)
else:
    print("Saved RQ4 localisation artifact is valid.")

=== RQ4 saved artifact validation ===
Descriptive method records: 3
McNemar pairwise comparisons: 3
Validation errors: 0
Saved RQ4 localisation artifact is valid.


In [ ]:
import json
from busi_xai.config import METRICS_DIR

# ---------------------------------------------------------------
# Load saved artifacts using their actual JSON structures
# ---------------------------------------------------------------

with open(
    METRICS_DIR / "xai_faithfulness_aggregate_metrics.json",
    "r"
) as f:
    faithfulness = json.load(f)

with open(
    METRICS_DIR / "xai_stability_aggregate_metrics.json",
    "r"
) as f:
    stability = json.load(f)

with open(
    METRICS_DIR / "xai_pairwise_agreement_aggregate_metrics.json",
    "r"
) as f:
    pairwise = json.load(f)

with open(
    METRICS_DIR / "test_classification_metrics.json",
    "r"
) as f:
    classification = json.load(f)


print("=== RQ4 EVIDENCE INVENTORY ===")


# ---------------------------------------------------------------
# 1. Localisation consistency
# ---------------------------------------------------------------

print("\n1. LOCALISATION CONSISTENCY")

for record in saved_rq4["descriptive_consistency"]:
    print(
        f"{record['method']}: "
        f"IoU mean={record['iou_mean']:.6f}, "
        f"IoU median={record['iou_median']:.6f}, "
        f"IoU non-zero={record['iou_nonzero_proportion']:.6f}, "
        f"IoU >= 0.25={record['iou_ge_0_25_proportion']:.6f}, "
        f"Dice mean={record['dice_mean']:.6f}"
    )


# ---------------------------------------------------------------
# 2. McNemar localisation consistency
# ---------------------------------------------------------------

print("\n2. MCNEMAR LOCALISATION CONSISTENCY")

for record in saved_rq4["mcnemar_results"]:
    print(
        f"{record['method_a']} vs {record['method_b']}: "
        f"p={record['p_value']:.6g}, "
        f"statistic={record['mcnemar_statistic']:.1f}"
    )


# ---------------------------------------------------------------
# 3. Faithfulness
# ---------------------------------------------------------------

print("\n3. FAITHFULNESS")

for method in ["gradcam", "lime", "shap"]:
    record = faithfulness[method]

    print(
        f"{method}: "
        f"mean probability drop="
        f"{record['probability_drop_mean']:.6f}"
    )


# ---------------------------------------------------------------
# 4. Stability
# ---------------------------------------------------------------

print("\n4. STABILITY")

for method in ["gradcam", "lime", "shap"]:
    record = stability[method]

    print(
        f"{method}: "
        f"mean Spearman="
        f"{record['spearman_mean']:.6f}, "
        f"mean MAE="
        f"{record['mae_mean']:.6f}"
    )


# ---------------------------------------------------------------
# 5. Pairwise explanation agreement
# ---------------------------------------------------------------

print("\n5. PAIRWISE EXPLANATION AGREEMENT")

for comparison in [
    "gradcam_vs_lime",
    "gradcam_vs_shap",
    "lime_vs_shap",
]:
    record = pairwise[comparison]

    print(
        f"{comparison}: "
        f"mean Spearman="
        f"{record['spearman_mean']:.6f}"
    )


# ---------------------------------------------------------------
# 6. Classification performance
# ---------------------------------------------------------------

print("\n6. CLASSIFICATION PERFORMANCE")

print(
    f"Test samples={classification['test_samples']}"
)

print(
    f"Accuracy={classification['accuracy']:.6f}"
)

print(
    f"Malignant precision="
    f"{classification['precision_malignant']:.6f}"
)

print(
    f"Malignant recall="
    f"{classification['recall_malignant']:.6f}"
)

print(
    f"Malignant F1="
    f"{classification['f1_malignant']:.6f}"
)

print(
    f"Benign specificity="
    f"{classification['specificity_benign']:.6f}"
)


print("\n=== RQ4 EVIDENCE INVENTORY COMPLETE ===")

=== RQ4 EVIDENCE INVENTORY ===

1. LOCALISATION CONSISTENCY
gradcam: IoU mean=0.190214, IoU median=0.164653, IoU non-zero=0.845361, IoU >= 0.25=0.329897, Dice mean=0.291171
lime: IoU mean=0.138400, IoU median=0.076100, IoU non-zero=0.907216, IoU >= 0.25=0.216495, Dice mean=0.213258
shap: IoU mean=0.176196, IoU median=0.157238, IoU non-zero=1.000000, IoU >= 0.25=0.319588, Dice mean=0.281676

2. MCNEMAR LOCALISATION CONSISTENCY
gradcam vs lime: p=0.263176, statistic=7.0
gradcam vs shap: p=6.10352e-05, statistic=0.0
lime vs shap: p=0.00390625, statistic=0.0

3. FAITHFULNESS
gradcam: mean probability drop=0.235403
lime: mean probability drop=0.136042
shap: mean probability drop=0.332149

4. STABILITY
gradcam: mean Spearman=0.674345, mean MAE=0.064652
lime: mean Spearman=0.377947, mean MAE=0.171704
shap: mean Spearman=0.646450, mean MAE=0.026270

5. PAIRWISE EXPLANATION AGREEMENT
gradcam_vs_lime: mean Spearman=0.231127
gradcam_vs_shap: mean Spearman=0.340051
lime_vs_shap: mean Spearman=0.16

In [ ]:
from pathlib import Path
import json

# ---------------------------------------------------------
# RQ4 evidence summary artifact
# ---------------------------------------------------------


rq4_localisation_path = METRICS_DIR / "xai_rq4_localisation_consistency.json"
faithfulness_path = METRICS_DIR / "xai_faithfulness_aggregate_metrics.json"
stability_path = METRICS_DIR / "xai_stability_aggregate_metrics.json"
pairwise_path = METRICS_DIR / "xai_pairwise_agreement_aggregate_metrics.json"
classification_path = METRICS_DIR / "test_classification_metrics.json"
statistics_path = METRICS_DIR / "xai_statistical_comparison.json"

output_path = METRICS_DIR / "xai_rq4_evidence_summary.json"


def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


# Load existing validated artifacts
rq4_localisation = load_json(rq4_localisation_path)
faithfulness = load_json(faithfulness_path)
stability = load_json(stability_path)
pairwise = load_json(pairwise_path)
classification = load_json(classification_path)
statistics = load_json(statistics_path)


# Keep only the statistical evidence relevant to RQ4
rq4_metrics = {
    "iou": None,
    "dice": None,
    "probability_drop": None,
    "stability_spearman": None,
    "stability_mae": None,
}

for result in statistics["friedman_results"]:
    metric = result["metric"]
    if metric in rq4_metrics:
        rq4_metrics[metric] = result


rq4_posthoc_metrics = {
    "probability_drop": [],
    "stability_spearman": [],
    "stability_mae": [],
}

for result in statistics["posthoc_results"]:
    metric = result["metric"]
    if metric in rq4_posthoc_metrics:
        rq4_posthoc_metrics[metric].append(result)


# Classification summary
classification_summary = {
    "test_samples": classification["test_samples"],
    "accuracy": classification["accuracy"],
    "malignant_precision": classification["precision_malignant"],
    "malignant_recall": classification["recall_malignant"],
    "malignant_f1": classification["f1_malignant"],
    "benign_specificity": classification["specificity_benign"],
}


# Structured RQ4 evidence profile.
# This deliberately avoids a composite score or unconditional winner.
evidence_profile = {
    "gradcam": {
        "localisation": "highest mean IoU and Dice among the three methods",
        "stability": "highest mean Spearman stability",
        "faithfulness": "intermediate mean probability drop",
    },
    "lime": {
        "localisation": "lowest mean IoU and Dice among the three methods",
        "stability": "lowest mean Spearman stability and highest mean MAE",
        "faithfulness": "lowest mean probability drop",
    },
    "shap": {
        "localisation": "highest non-zero overlap proportion and second-highest mean IoU and Dice",
        "stability": "lowest mean MAE and second-highest mean Spearman stability",
        "faithfulness": "highest mean probability drop",
    },
}


summary = {
    "design": {
        "research_question": "RQ4",
        "question": (
            "Which of the evaluated XAI methods provides the most consistent "
            "evidence of clinically relevant model attention under the defined "
            "experimental conditions?"
        ),
        "n_test_images": 97,
        "methods": ["gradcam", "lime", "shap"],
        "evidence_is_multidimensional": True,
        "composite_score_used": False,
        "unconditional_single_winner_claim": False,
    },

    "localisation_consistency": {
        "descriptive_consistency": rq4_localisation["descriptive_consistency"],
        "mcnemar_results": rq4_localisation["mcnemar_results"],
    },

    "faithfulness": faithfulness,

    "stability": stability,

    "pairwise_explanation_agreement": pairwise,

    "classification_performance": classification_summary,

    "inferential_comparison": {
        "friedman_results": rq4_metrics,
        "posthoc_results": rq4_posthoc_metrics,
    },

    "evidence_profile": evidence_profile,
}


# Save
with open(output_path, "w") as f:
    json.dump(summary, f, indent=2)


print("=== RQ4 evidence summary saved ===")
print(f"Path: {output_path}")
print(f"Methods: {summary['design']['methods']}")
print(f"Test images: {summary['design']['n_test_images']}")
print(f"Composite score used: {summary['design']['composite_score_used']}")
print(f"Unconditional single winner claim: {summary['design']['unconditional_single_winner_claim']}")
print("Evidence sections:")
print(f"- Localisation: {len(summary['localisation_consistency']['descriptive_consistency'])} methods")
print(f"- McNemar comparisons: {len(summary['localisation_consistency']['mcnemar_results'])}")
print(f"- Faithfulness: {len(summary['faithfulness'])} methods")
print(f"- Stability: {len(summary['stability'])} methods")
print(f"- Pairwise agreement: {len(summary['pairwise_explanation_agreement'])}")
print(f"- Friedman metrics: {len(summary['inferential_comparison']['friedman_results'])}")
print(f"- Post-hoc metrics: {len(summary['inferential_comparison']['posthoc_results'])}")

=== RQ4 evidence summary saved ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_rq4_evidence_summary.json
Methods: ['gradcam', 'lime', 'shap']
Test images: 97
Composite score used: False
Unconditional single winner claim: False
Evidence sections:
- Localisation: 3 methods
- McNemar comparisons: 3
- Faithfulness: 3 methods
- Stability: 3 methods
- Pairwise agreement: 3
- Friedman metrics: 5
- Post-hoc metrics: 3


In [ ]:
import json
from pathlib import Path

summary_path = Path(
    "/content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_rq4_evidence_summary.json"
)

with open(summary_path, "r") as f:
    summary = json.load(f)

errors = []

# ---------------------------------------------------------
# Basic design validation
# ---------------------------------------------------------
if summary["design"]["research_question"] != "RQ4":
    errors.append("Research question is not RQ4.")

if summary["design"]["n_test_images"] != 97:
    errors.append("Unexpected number of test images.")

expected_methods = ["gradcam", "lime", "shap"]

if summary["design"]["methods"] != expected_methods:
    errors.append("Unexpected method list.")

if summary["design"]["composite_score_used"] is not False:
    errors.append("Composite score flag must be False.")

if summary["design"]["unconditional_single_winner_claim"] is not False:
    errors.append("Unconditional single-winner flag must be False.")


# ---------------------------------------------------------
# Evidence-section validation
# ---------------------------------------------------------
if len(summary["localisation_consistency"]["descriptive_consistency"]) != 3:
    errors.append("Expected 3 localisation method records.")

if len(summary["localisation_consistency"]["mcnemar_results"]) != 3:
    errors.append("Expected 3 McNemar comparisons.")

if set(summary["faithfulness"].keys()) != set(expected_methods):
    errors.append("Faithfulness methods do not match expected methods.")

if set(summary["stability"].keys()) != set(expected_methods):
    errors.append("Stability methods do not match expected methods.")

if len(summary["pairwise_explanation_agreement"]) != 3:
    errors.append("Expected 3 pairwise agreement comparisons.")


# ---------------------------------------------------------
# Statistical validation
# ---------------------------------------------------------
friedman = summary["inferential_comparison"]["friedman_results"]

expected_friedman_metrics = {
    "iou",
    "dice",
    "probability_drop",
    "stability_spearman",
    "stability_mae",
}

if set(friedman.keys()) != expected_friedman_metrics:
    errors.append("Unexpected Friedman metric set.")

posthoc = summary["inferential_comparison"]["posthoc_results"]

expected_posthoc_metrics = {
    "probability_drop",
    "stability_spearman",
    "stability_mae",
}

if set(posthoc.keys()) != expected_posthoc_metrics:
    errors.append("Unexpected post-hoc metric set.")


# ---------------------------------------------------------
# Required numerical evidence checks
# ---------------------------------------------------------
if abs(
    summary["faithfulness"]["gradcam"]["probability_drop_mean"]
    - 0.2354026720043003
) > 1e-12:
    errors.append("Grad-CAM faithfulness value does not match saved evidence.")

if abs(
    summary["faithfulness"]["lime"]["probability_drop_mean"]
    - 0.13604225919188298
) > 1e-12:
    errors.append("LIME faithfulness value does not match saved evidence.")

if abs(
    summary["faithfulness"]["shap"]["probability_drop_mean"]
    - 0.33214866154894385
) > 1e-12:
    errors.append("SHAP faithfulness value does not match saved evidence.")


if errors:
    print("=== RQ4 evidence summary validation FAILED ===")
    for error in errors:
        print(f"- {error}")
else:
    print("=== RQ4 evidence summary validation ===")
    print("Research question: RQ4")
    print("Test images:", summary["design"]["n_test_images"])
    print("Methods:", summary["design"]["methods"])
    print(
        "Composite score used:",
        summary["design"]["composite_score_used"]
    )
    print(
        "Unconditional single winner claim:",
        summary["design"]["unconditional_single_winner_claim"]
    )
    print(
        "Localisation method records:",
        len(summary["localisation_consistency"]["descriptive_consistency"])
    )
    print(
        "McNemar comparisons:",
        len(summary["localisation_consistency"]["mcnemar_results"])
    )
    print(
        "Friedman metrics:",
        len(summary["inferential_comparison"]["friedman_results"])
    )
    print(
        "Post-hoc metrics:",
        len(summary["inferential_comparison"]["posthoc_results"])
    )
    print("Validation errors:", len(errors))
    print("Saved RQ4 evidence summary is valid.")

=== RQ4 evidence summary validation ===
Research question: RQ4
Test images: 97
Methods: ['gradcam', 'lime', 'shap']
Composite score used: False
Unconditional single winner claim: False
Localisation method records: 3
McNemar comparisons: 3
Friedman metrics: 5
Post-hoc metrics: 3
Validation errors: 0
Saved RQ4 evidence summary is valid.


# Checks

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/BUSI-XAI")

RESULTS_EVIDENCE_MAP_PATH = (
    PROJECT_ROOT / "outputs/metrics/results_evidence_map.json"
)

results_evidence_map = {
    "status": "prepared_for_dissertation_results",

    "classification": {
        "test_images": 97,
        "accuracy": 0.8969072164948454,
        "malignant_precision": 0.8387096774193549,
        "malignant_recall": 0.8387096774193549,
        "malignant_f1": 0.8387096774193549,
        "benign_specificity": 0.9242424242424242,
        "checkpoint_epoch": 8,
        "best_validation_loss": 0.314631738436988,
    },

    "RQ1": {
        "question": (
            "How do Grad-CAM, LIME and SHAP differ in their explanations "
            "of predictions generated by the same medical image "
            "classification model under controlled experimental conditions?"
        ),
        "primary_evidence": [
            "xai_records.json",
            "xai_pairwise_agreement_aggregate_metrics.json",
            "xai_runtime_aggregate_metrics.json",
        ],
        "descriptive_metrics": {
            "pairwise_spearman": {
                "gradcam_lime": 0.231127,
                "gradcam_shap": 0.340051,
                "lime_shap": 0.162949,
            },
            "mean_runtime_seconds": {
                "gradcam": 0.025664,
                "lime": 9.827661,
                "shap": 2.039720,
            },
        },
    },

    "RQ2": {
        "question": (
            "How do Grad-CAM, LIME and SHAP compare according to the "
            "selected quantitative measures of explanation quality, "
            "including spatial agreement with available ground-truth regions?"
        ),
        "primary_evidence": [
            "xai_aggregate_metrics.json",
            "xai_faithfulness_aggregate_metrics.json",
            "xai_stability_aggregate_metrics.json",
            "xai_statistical_comparison.json",
        ],
        "localisation": {
            "gradcam": {
                "iou_mean": 0.190214,
                "dice_mean": 0.291171,
            },
            "lime": {
                "iou_mean": 0.138400,
                "dice_mean": 0.213258,
            },
            "shap": {
                "iou_mean": 0.176196,
                "dice_mean": 0.281676,
            },
        },
        "faithfulness_probability_drop": {
            "gradcam": 0.2354026720043003,
            "lime": 0.13604225919188298,
            "shap": 0.33214866154894385,
        },
        "stability_spearman": {
            "gradcam": 0.6743454702465592,
            "lime": 0.3779471869427136,
            "shap": 0.6464496657145121,
        },
        "stability_mae": {
            "gradcam": 0.06465208689807002,
            "lime": 0.17170370377830624,
            "shap": 0.026270356710961798,
        },
        "friedman": {
            "iou": {
                "statistic": 5.119171,
                "p": 0.0773368,
            },
            "dice": {
                "statistic": 5.119171,
                "p": 0.0773368,
            },
            "probability_drop": {
                "statistic": 38.185567,
                "p": 5.106337e-09,
            },
            "stability_spearman": {
                "statistic": 96.577320,
                "p": 1.067829e-21,
            },
            "stability_mae": {
                "statistic": 169.257732,
                "p": 1.762587e-37,
            },
            "runtime_seconds": {
                "statistic": 194.0,
                "p": 7.471972e-43,
            },
        },
        "interpretation_constraints": [
            "Overall Friedman tests for IoU and Dice were not significant at alpha=0.05.",
            "Do not claim statistically significant overall localisation differences.",
            "Faithfulness differences were statistically significant.",
            "Stability differences were statistically significant.",
            "Runtime differences were statistically significant.",
        ],
    },

    "RQ3": {
        "question": (
            "What relationship exists between classification performance "
            "and the quality of the explanations produced by the three XAI methods?"
        ),
        "primary_evidence": [
            "xai_rq3_classification_explanation_relationship.json"
        ],
        "joined_rows": 291,
        "test_images": 97,
        "correct_classifications": 261,
        "incorrect_classifications": 30,
        "correlations": {
            "gradcam": {
                "iou": {"rho": 0.072720, "p": 4.790324e-01},
                "dice": {"rho": 0.072720, "p": 4.790324e-01},
                "probability_drop": {
                    "rho": -0.294275,
                    "p": 3.436008e-03,
                },
                "stability_spearman": {
                    "rho": 0.707316,
                    "p": 5.582055e-16,
                },
            },
            "lime": {
                "iou": {"rho": 0.235773, "p": 2.007886e-02},
                "dice": {"rho": 0.235773, "p": 2.007886e-02},
                "probability_drop": {
                    "rho": -0.014767,
                    "p": 8.858476e-01,
                },
                "stability_spearman": {
                    "rho": 0.059081,
                    "p": 5.653991e-01,
                },
            },
            "shap": {
                "iou": {"rho": 0.106183, "p": 3.005973e-01},
                "dice": {"rho": 0.106183, "p": 3.005973e-01},
                "probability_drop": {
                    "rho": 0.332566,
                    "p": 8.740807e-04,
                },
                "stability_spearman": {
                    "rho": 0.232011,
                    "p": 2.221204e-02,
                },
            },
        },
    },

    "RQ4": {
        "question": (
            "Which of the evaluated XAI methods provides the most consistent "
            "evidence of clinically relevant model attention under the defined "
            "experimental conditions?"
        ),
        "primary_evidence": [
            "xai_rq4_localisation_consistency.json",
            "xai_rq4_evidence_summary.json",
        ],
        "localisation_success": {
            "gradcam_iou_nonzero": 0.845361,
            "lime_iou_nonzero": 0.907216,
            "shap_iou_nonzero": 1.000000,
        },
        "mcnemar": {
            "gradcam_vs_lime_p": 0.263176,
            "gradcam_vs_shap_p": 0.000061,
            "lime_vs_shap_p": 0.003906,
        },
        "interpretation_constraints": [
            "Do not claim an unconditional overall winner.",
            "Grad-CAM shows the strongest continuous localisation mean and highest stability Spearman.",
            "SHAP shows the strongest faithfulness, lowest stability MAE, and complete non-zero localisation consistency.",
            "LIME is generally weaker across the quantitative evidence.",
            "The evidence is multidimensional rather than reducible to one overall ranking.",
        ],
    },

    "global_methodological_constraints": [
        "One dataset and one trained ResNet-50 classifier were used.",
        "The same 97 test images were used across methods.",
        "The original predicted class was retained as the explanation target.",
        "The established 90th-percentile localisation threshold was retained.",
        "No composite score was used.",
        "No unconditional single winner was established.",
        "Results describe the defined experimental conditions and should not be generalised beyond them without qualification.",
    ],
}

with open(RESULTS_EVIDENCE_MAP_PATH, "w") as f:
    json.dump(results_evidence_map, f, indent=2)

# Basic validation
with open(RESULTS_EVIDENCE_MAP_PATH, "r") as f:
    saved_map = json.load(f)

assert saved_map["status"] == "prepared_for_dissertation_results"
assert set(saved_map["RQ1"].keys()) >= {"question", "primary_evidence"}
assert set(saved_map["RQ2"].keys()) >= {"question", "primary_evidence"}
assert set(saved_map["RQ3"].keys()) >= {"question", "primary_evidence"}
assert set(saved_map["RQ4"].keys()) >= {"question", "primary_evidence"}

assert saved_map["RQ2"]["friedman"]["iou"]["p"] > 0.05
assert saved_map["RQ2"]["friedman"]["dice"]["p"] > 0.05

assert (
    "Do not claim an unconditional overall winner."
    in saved_map["RQ4"]["interpretation_constraints"]
)

print("=== Results evidence map validation ===")
print("RQ1 evidence: present")
print("RQ2 evidence: present")
print("RQ3 evidence: present")
print("RQ4 evidence: present")
print("Non-significant localisation caution: recorded")
print("No composite-score interpretation: recorded")
print("No unconditional-winner interpretation: recorded")
print(f"Saved: {RESULTS_EVIDENCE_MAP_PATH}")
print("Results evidence map is valid.")

=== Results evidence map validation ===
RQ1 evidence: present
RQ2 evidence: present
RQ3 evidence: present
RQ4 evidence: present
Non-significant localisation caution: recorded
No composite-score interpretation: recorded
No unconditional-winner interpretation: recorded
Saved: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/results_evidence_map.json
Results evidence map is valid.


In [ ]:
from pathlib import Path
import json
import math

METRICS_DIR = Path("/content/drive/MyDrive/BUSI-XAI/outputs/metrics")


def load_json(filename):
    with open(METRICS_DIR / filename, "r") as f:
        return json.load(f)


# ------------------------------------------------------------------
# Load saved artifacts
# ------------------------------------------------------------------

evidence = load_json("results_evidence_map.json")
classification = load_json("test_classification_metrics.json")
localisation = load_json("xai_aggregate_metrics.json")
faithfulness = load_json("xai_faithfulness_aggregate_metrics.json")
stability = load_json("xai_stability_aggregate_metrics.json")
pairwise = load_json("xai_pairwise_agreement_aggregate_metrics.json")
runtime = load_json("xai_runtime_aggregate_metrics.json")
rq3 = load_json("xai_rq3_classification_explanation_relationship.json")
statistics = load_json("xai_statistical_comparison.json")

errors = []


def check_close(label, mapped_value, artifact_value, tol=1e-6):
    if mapped_value is None:
        errors.append(f"{label}: missing in evidence map")
        return

    if not math.isclose(
        float(mapped_value),
        float(artifact_value),
        rel_tol=tol,
        abs_tol=tol,
    ):
        errors.append(
            f"{label}: map={mapped_value}, artifact={artifact_value}"
        )


# ------------------------------------------------------------------
# 1. Classification
# ------------------------------------------------------------------

c = evidence["classification"]

check_close("classification accuracy",
            c["accuracy"],
            classification["accuracy"])

check_close("classification malignant precision",
            c["malignant_precision"],
            classification["precision_malignant"])

check_close("classification malignant recall",
            c["malignant_recall"],
            classification["recall_malignant"])

check_close("classification malignant F1",
            c["malignant_f1"],
            classification["f1_malignant"])

check_close("classification benign specificity",
            c["benign_specificity"],
            classification["specificity_benign"])


# ------------------------------------------------------------------
# 2. RQ1: pairwise agreement + runtime
# ------------------------------------------------------------------

pairwise_map = {
    "gradcam_lime": "gradcam_vs_lime",
    "gradcam_shap": "gradcam_vs_shap",
    "lime_shap": "lime_vs_shap",
}

for map_key, artifact_key in pairwise_map.items():

    mapped = evidence["RQ1"]["descriptive_metrics"]["pairwise_spearman"][map_key]
    artifact = pairwise[artifact_key]

    check_close(
        f"RQ1 pairwise {map_key}",
        mapped,
        artifact["spearman_mean"],
    )


for method in ["gradcam", "lime", "shap"]:

    mapped = evidence["RQ1"]["descriptive_metrics"]["mean_runtime_seconds"][method]
    artifact = runtime[method]

    check_close(
        f"RQ1 runtime {method}",
        mapped,
        artifact["runtime_mean_seconds"],
    )


# ------------------------------------------------------------------
# 3. RQ2: localisation
# ------------------------------------------------------------------

for method in ["gradcam", "lime", "shap"]:

    mapped = evidence["RQ2"]["localisation"][method]
    artifact = localisation[method]["overall"]

    check_close(
        f"RQ2 localisation {method} IoU",
        mapped["iou_mean"],
        artifact["iou_mean"],
    )

    check_close(
        f"RQ2 localisation {method} Dice",
        mapped["dice_mean"],
        artifact["dice_mean"],
    )


# ------------------------------------------------------------------
# 4. RQ2: faithfulness
# ------------------------------------------------------------------

for method in ["gradcam", "lime", "shap"]:

    mapped = evidence["RQ2"]["faithfulness_probability_drop"][method]
    artifact = faithfulness[method]["probability_drop_mean"]

    check_close(
        f"RQ2 faithfulness {method}",
        mapped,
        artifact,
    )


# ------------------------------------------------------------------
# 5. RQ2: stability
# ------------------------------------------------------------------

for method in ["gradcam", "lime", "shap"]:

    mapped_spearman = evidence["RQ2"]["stability_spearman"][method]
    mapped_mae = evidence["RQ2"]["stability_mae"][method]

    artifact_spearman = stability[method]["spearman_mean"]
    artifact_mae = stability[method]["mae_mean"]

    check_close(
        f"RQ2 stability {method} Spearman",
        mapped_spearman,
        artifact_spearman,
    )

    check_close(
        f"RQ2 stability {method} MAE",
        mapped_mae,
        artifact_mae,
    )

# ------------------------------------------------------------------
# 6. RQ2: Friedman statistics
# ------------------------------------------------------------------

artifact_friedman = {
    record["metric"]: record
    for record in statistics["friedman_results"]
}

for metric, mapped in evidence["RQ2"]["friedman"].items():

    if metric not in artifact_friedman:
        errors.append(
            f"RQ2 Friedman: {metric} missing from artifact"
        )
        continue

    artifact = artifact_friedman[metric]

    check_close(
        f"RQ2 Friedman {metric} statistic",
        mapped["statistic"],
        artifact["friedman_statistic"],
        tol=1e-5,
    )

    check_close(
        f"RQ2 Friedman {metric} p-value",
        mapped["p"],
        artifact["p_value"],
        tol=1e-5,
    )
# ------------------------------------------------------------------
# 7. RQ3: sample counts
# ------------------------------------------------------------------

rq3_map = evidence["RQ3"]

if rq3_map["joined_rows"] != rq3["joined_rows"]:
    errors.append(
        f"RQ3 joined rows mismatch: "
        f"map={rq3_map['joined_rows']}, "
        f"artifact={rq3['joined_rows']}"
    )

if rq3_map["test_images"] != rq3["dataset_size"]:
    errors.append(
        f"RQ3 test images mismatch: "
        f"map={rq3_map['test_images']}, "
        f"artifact={rq3['dataset_size']}"
    )

if rq3_map["correct_classifications"] != rq3["classification_counts"]["correct"]:
    errors.append(
        f"RQ3 correct classifications mismatch: "
        f"map={rq3_map['correct_classifications']}, "
        f"artifact={rq3['classification_counts']['correct']}"
    )

if rq3_map["incorrect_classifications"] != rq3["classification_counts"]["incorrect"]:
    errors.append(
        f"RQ3 incorrect classifications mismatch: "
        f"map={rq3_map['incorrect_classifications']}, "
        f"artifact={rq3['classification_counts']['incorrect']}"
    )

# ------------------------------------------------------------------
# 8. RQ3: correlations
# ------------------------------------------------------------------

artifact_correlations = {
    (record["method"], record["metric"]): record
    for record in rq3["probability_correlations"]
}

for method, metrics in evidence["RQ3"]["correlations"].items():

    for metric, mapped in metrics.items():

        key = (method, metric)

        if key not in artifact_correlations:
            errors.append(
                f"RQ3 correlation missing: {method}/{metric}"
            )
            continue

        artifact = artifact_correlations[key]

        check_close(
            f"RQ3 {method} {metric} rho",
            mapped["rho"],
            artifact["spearman_rho"],
            tol=1e-5,
        )

        check_close(
            f"RQ3 {method} {metric} p-value",
            mapped["p"],
            artifact["p_value"],
            tol=1e-5,
        )

# ------------------------------------------------------------------
# Final validation
# ------------------------------------------------------------------

print("=== Results evidence source-of-truth verification ===")
print(f"Validation errors: {len(errors)}")

if errors:
    print("\nERRORS:")
    for error in errors:
        print("-", error)

    raise AssertionError(
        "Evidence map does not fully match the saved experiment artifacts."
    )

print("\nEvidence map matches the saved experiment artifacts.")
print("Source-of-truth verification passed.")

=== Results evidence source-of-truth verification ===
Validation errors: 0

Evidence map matches the saved experiment artifacts.
Source-of-truth verification passed.


In [13]:
%run /content/drive/MyDrive/BUSI-XAI/experiment-runner/run_experiment.py

BUSI-XAI EXPERIMENT RUNNER — TEST

[1/6] Running step_01_preprocessing.py
step 1
PASSED: step_01_preprocessing.py

[2/6] Running step_02_data_splitting.py
step 2
PASSED: step_02_data_splitting.py

[3/6] Running step_03_model_training.py
step 3
PASSED: step_03_model_training.py

[4/6] Running step_04_classification_evaluation.py
step 4
PASSED: step_04_classification_evaluation.py

[5/6] Running step_05_xai_generation.py
step 5
PASSED: step_05_xai_generation.py

[6/6] Running step_06_xai_evaluation.py
step 6
PASSED: step_06_xai_evaluation.py

ALL SIX STEP FILES PASSED THE TEST
